# Analysis on the Relationship Between Resource Usage and Temperature on Raspberry Pi Devices

Link to Github page: [https://github.com/CuriousChum/jsc370](https://github.com/CuriousChum/jsc370)

## Introduction

### Background
Edge computing devices such as Raspberry Pis have started gaining popularity due to their minimal design and affordable price tag. Unlike their bulkier counterpart, Raspberry Pis are small, yet powerful enough to act as a miniature personal computer (PC). This makes them suitable for specialized tasks such as gathering field data. However, due to their size and the rising demand in performance, the temperature within the device motherboard can rise quickly. The motherboard, being in the center of the device, is affected by the activity of peripheral devices and may wear out from intensive use. 

This analysis aims explores the relationship between device resources utilization and temperature across different Raspberry Pi specifications. The goal is to understand how different workloads affect device temperature (as measured by sensors in the motherboard) and whether this relationship can be used to predict thermal behavior under various application usages and resource utilization. This information is valuable for optimizing edge computing applications by anticipating device wear out and CPU throttling due to overheating.

### Research Question

How does resource utilization affect device temperature across different Raspberry Pi configurations and application states, and can we identify patterns in this relationship to predict overall device temperature?

## Methods

### Dataset Description

The dataset used in this analysis was constructed and collected by researched is the Queen's Telecommunation Research Lab (TRL), led by Ruslan Kain. It contains resource usage information from four heterogeneous Raspberry Pi 4 devices, with different RAM sizes (2GB, 4GB, 8GB) and CPU frequencies (1200MHz, 1500MHz, 1800MHz), measured under different application usages including gaming, streaming, augmented reality, mining, and idling. The data is collected in ~5 second intervals using the `PsUtil` Python package.

The dataset includes measurements from usages with random and periodic workloads using different Raspberry Pis. Each `.tab` file in the dataset contains data on a 'run' of a device. The tables are denoted as `<device specification>_res_usage_data_*.tab`. The runs included in this analysis are only from files that matches the following pattern: `<device specification>_res_usage_data_rvp_*.tab`.

{numref}`tbl-vars` lists all the variables regarding to device resource information during each run of the Raspberry Pi 4s that is given in the dataset.

```{table} Variable Descriptions
:name: tbl-vars

| Variable Name         | Description                                                                                                                              |
|-----------------------|------------------------------------------------------------------------------------------------------------------------------------------|
| `time_stamp`          | Precise wall-clock time when the measurement was taken                                                                                    |
| `time`                | Time as a floating-point number expressed in seconds since the Unix epoch (UTC)                                                          |
| `cpu_freq`            | System-wide CPU cycle frequency (MHz)                                                                                                    |
| `cpu`                 | System-wide CPU utilization as a percentage (%)                                                                                          |
| `cpu_user_time`       | Time spent by normal processes executing in user mode, including guest time (seconds)                                                    |
| `cpu_idle_time`       | Time spent doing nothing (seconds)                                                                                                       |
| `cpu_system_time`     | Time spent by processes executing in kernel mode (seconds)                                                                               |
| `memory`              | Memory currently in use or very recently used, as a percentage of RAM (%)                                                                |
| `net_sent`            | Total bytes sent since boot                                                                                                              |
| `net_recv`            | Total bytes received since boot                                                                                                          |
| `net_upload_rate`     | Bytes sent in the last interval, divided by the interval length (MB/s)                                                                   |
| `net_download_rate`   | Bytes received in the last interval, divided by the interval length (MB/s)                                                               |
| `temp`                | Device temperature (°C)                                                                                                                  |
| `wifi_freq`           | Operating frequency / channel of the Wi-Fi interface (GHz)                                                                              |
| `bit_rate`            | Current Wi-Fi bit rate (bit/s)                                                                                                           |
| `link_quality`        | Overall Wi-Fi link-quality metric (contention, interference, error rate, etc.)                                                          |
| `link_quality_max`    | Maximum possible value of the Wi-Fi link-quality metric                                                                                 |
| `gpu`                 | GPU utilization as a percentage (%)                                                                                                      |
| `gpu_memory`          | GPU memory in use or very recently used (%)                                                                                              |
| `state`               | Resource-usage state of the device associated with the running application (e.g., *mining*, *streaming*, *idle*)                        |
```

### Data Wrangling

The dataset contains a lot of columns, some of which are redundant, filled with only zeros, or describes irrelevant information; these columns will thus be dropped from the analysis.

First, columns describing CPU time spent will be dropped since the focus is mainly on overall CPU utilization, so `cpu` and `cpu_freq` is sufficient. Manual inspection of the data shows that the columns `gpu` and `gpu_memory` are always zero, implying that the builtin GPU is not utilized in any of the workloads, so it is drop. Furthermore, the columns `link_quality_max` and `link_quality` are dropped because of the lack of documentation; the exact descriptions cannot be found in the `PsUtil` package. Finally, the columns `wifi_freq`, `net_sent`, and `net_recv` are dropped because these are already better represented by `net_upload_rate` and `net_download_rate`; note that `wifi_freq` is not indicative of device resource usage and both `net_*` features are cumulative sums of the `net_*_rate`s, respectively.

Next, the tables from the dataset will be combined while preserving per-device and per-run information using the columns `pi_id` and `run_id`, respectively. For brevity, the devices will be denoted as follows

- `dev0`: Raspberry Pi 4, 2GB RAM, 1200 MHz
- `dev1`: Raspberry Pi 4, 2GB RAM, 1500 MHz
- `dev2`: Raspberry Pi 4, 4GB RAM, 1500 MHz
- `dev3`: Raspberry Pi 4, 8GB RAM, 1800 MHz

Meanwhile, `run_id` ranges from `000` to `018`, each table in the initial dataset is considered a different run, irrespective of the device.

The final pruned dataset contains columns that is described in {numref}`tbl-vars-pruned`.  
```{table} Pruned Variable Descriptions
:name: tbl-vars-pruned
| Variable Name       | Description                                                                                      |
|---------------------|--------------------------------------------------------------------------------------------------|
| `cpu`               | System-wide CPU utilization as a percentage (%)                                                  |
| `cpu_freq`          | System-wide CPU cycle frequency (MHz)                                                            |
| `state`             | Label feature for the resource-usage state of the device associated with the running application |
| `memory`            | Memory currently in use or very recently used, as a percentage of RAM (%)                        |
| `net_upload_rate`   | Number of bytes sent in the last time interval, divided by the time interval (MBytes/s)          |
| `net_download_rate` | Number of bytes received in the last time interval, divided by the time interval (MBytes/s)      |
| `temp`              | Device temperature (°C)                                                                          |
| `pi_id`              | Device id, one of `dev0`, `dev1`, `dev2`, `dev3` |
| `run_id`              | Run id, ranges from `000` to `018` |
```


In [2]:
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.formula.api as smf

from scipy.stats import chi2
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error as mse
from xgboost import XGBRegressor

import math
import importlib
import gc

import utils

import warnings
warnings.filterwarnings('ignore')

In [3]:
url = "https://borealisdata.ca/api/access/dataset/:persistentId"
rpi_id = "doi:10.5683/SP3/GOZAJE"
params = {
    "persistentId": rpi_id,
}
force = False
if force:
    utils.get_data_from_remote(url, params)

In [4]:
COLS = [
    'time_stamp', 'time', 'state', 'cpu', 'cpu_freq',
    'memory', 'net_upload_rate', 'net_download_rate', 'temp',
    'gpu'
]
fnames = utils.get_filenames()

runs = []
for i, f in enumerate(fnames):
    runs.append(utils.load_one(f, run_id=f"{i:03d}", cols=COLS))

df_all = pl.concat(runs, how="diagonal")
states = np.sort(df_all["state"].unique())

### Exploratory Data Analysis

Since each run is a time-series data, comparing each feature directly might not be ideal. Instead, the comparison will either be done over time, or on the differences.

#### Comparison over Time

Since it's infeasible to visualize and compare all runs with more than 30,000 observations, we instead sample one run from different devices and look at random sections within it. {numref}`fig-time0`, {numref}`fig-time1`, {numref}`fig-time2`, {numref}`fig-time3` are sections of runs that are chosen uniformly at random for each device.

In [5]:
np.random.seed(1000)
chosen = {}
for dev, runs in df_all[["pi_id", "run_id"]].unique().group_by("pi_id"):
    chosen[dev[0]] = np.random.choice(runs["run_id"], size=1)[0]

In [38]:
def sample_section(df):
    n_take = math.ceil(len(df) * 0.1)
    start = np.random.randint(0, len(df) - n_take)  
    return df[start:start + n_take - 1] # be safe, don't crash for no reason

def plot_dev_mult_time_series(dev):
    plotdf = sample_section(
        df_all.filter(
            (pl.col("pi_id") == dev) & (pl.col("run_id") == chosen[dev])
        )
    )
    return utils.plot_mult_time_series(
        plotdf,
        ['cpu', "cpu_freq", 'temp', "memory", "net_upload_rate", "net_download_rate"],
        ylabs=['CPU usage (%)', 'CPU frequency', 'Temperature (Celcius)', "Memory",
            "Net Upload Rate (MBps)", "Net Download Rate (MBps)"
              ],
        suptitle=f'Resource Activity and Temperature over Time for device {dev} on run {chosen[dev]}'
    )


In [40]:
fig, axs = plot_dev_mult_time_series('dev0')
fig.savefig("images/fig-time0.png")
plt.close(fig)
fig, axs = plot_dev_mult_time_series('dev1')
fig.savefig("images/fig-time1.png")
plt.close(fig)
fig, axs = plot_dev_mult_time_series('dev2')
fig.savefig("images/fig-time2.png")
plt.close(fig)
fig, axs = plot_dev_mult_time_series('dev3')
fig.savefig("images/fig-time3.png")
plt.close(fig)

```{figure} images/fig-time0.png
:name: fig-time0
:align: center
:scale: 40
plots of several resource activity samples for `dev0`
```
```{figure} images/fig-time1.png
:name: fig-time1
:align: center
:scale: 40
plots of several resource activity samples for `dev1`
```
```{figure} images/fig-time2.png
:name: fig-time2
:align: center
:scale: 40
plots of several resource activity samples for `dev2`
```
```{figure} images/fig-time3.png
:name: fig-time3
:align: center
:scale: 40
plots of several resource activity samples for `dev3`
```

The plots shows several notable things: device temperature tends to move to a certain semi-constant value on each state but with a noticable jitter. Similarly, CPU usage tends to move to a constant value with noticable jitters and occasional spikes every state. CPU frequency, on the other hand, appears to erratically jump from high to low, this is a phenomenon called throttling, where the CPU dynamically changes its frequency due to some circumstance, either due to low usage (as to preserve energy), or due to overheating, which is not seen here. Finally, we observe that there is only network activity when the device is streaming. Finally, memory usage is consistent within states with a more noticable variation when the device is streaming or running an augmented reality application.

#### Feature Distributions

Then, we visualize the distribution of features per device colored by state. The following figures shows the aggregated distribution over all runs. Network upload/download rates are scaled up to the scale of KBps in this visualization due to be more noticable.

In [8]:
df_all = df_all.with_columns(
    (pl.col("net_upload_rate") * 1000).alias("net_upload_rate_kbps"),
    (pl.col("net_download_rate") * 1000).alias("net_download_rate_kbps")
)
def plot_activity(df, dev, cols, ylabs=None, suptitle=None):
    fig, ax = plt.subplots(ncols=1, nrows=len(cols), figsize=(12, 2 * len(cols)))
    ylabs = ylabs if ylabs is not None else cols
    
    legend_handles, legend_labels = None, None
    
    for idx, col in enumerate(cols):
        sns.histplot(df, x=col, hue='state', ax=ax[idx], bins=50, hue_order=states, kde=True)
        ax[idx].set_xlabel('count')
        ax[idx].set_ylabel(ylabs[idx])
        if idx != len(cols) - 1:
            ax[idx].get_legend().remove()
        
    fig.suptitle(f'Resource Activity Distribution in {dev}')
    fig.tight_layout()
    return fig, ax

cols = ["cpu", "cpu_freq", "memory", "temp", "net_upload_rate_kbps", "net_download_rate_kbps"]
fig, ax = plot_activity(
    df_all.filter(pl.col("pi_id") == "dev0"), "dev0", cols)
fig.savefig("images/pa0.png")
plt.close(fig)
fig, ax = plot_activity(
    df_all.filter(pl.col("pi_id") == "dev1"), "dev1", cols)
fig.savefig("images/pa1.png")
plt.close(fig)
fig, ax = plot_activity(
    df_all.filter(pl.col("pi_id") == "dev2"), "dev2", cols)
fig.savefig("images/pa2.png")
plt.close(fig)
fig, ax = plot_activity(
    df_all.filter(pl.col("pi_id") == "dev3"), "dev3", cols)
fig.savefig("images/pa3.png")
plt.close(fig)

```{figure} images/pa0.png
:name: fig-pa0
:align: center
:scale: 40

resource activity distribution for `dev0`
```
```{figure} images/pa1.png
:name: fig-pa1
:align: center
:scale: 40

resource activity distribution for `dev1`
```
```{figure} images/pa2.png
:name: fig-pa2
:align: center
:scale: 40

resource activity distribution for `dev2`
```
```{figure} images/pa3.png
:name: fig-pa3
:align: center
:scale: 40

resource activity distribution for `dev3`
```

There are notable patterns in the distribution of each state: temperature and CPU usage is approximately unimodal for each state; CPU frequency in each state are either maximum or a specific value with not much in between; memory usage being quite consistent with similar centers, albeit having different spread/variance between device; network is activity is dominantly zero.

#### Intra-Feature Relationship 

Looking at several relationships between features, we see consistent relationship as seen before. Most notably, the clusters when plotting CPU usage vs. temperature as seen below in {numref}`fig-ct`.

In [9]:
dev = "dev3"
plotdf = df_all.filter(
        (pl.col("pi_id") == dev) & (pl.col("run_id") == chosen[dev])
)
sns.scatterplot(plotdf, x='cpu', y='temp', hue='state')

plt.gca().set_title("CPU Usage vs. Temperature")
plt.gcf().savefig("images/cpu_v_temp.png")
plt.close()

```{figure} images/cpu_v_temp.png
:name: fig-ct
:align: center
:scale: 60

CPU usage vs. Temperature on `dev3`
```

#### Feature Engineering

Motivated by findings in the EDA phase, we will create the following columns as a means to transform the original features:
- `cpu_freq_mean`: the average CPU frequency within a period of a state in each run per device. Since the CPU throttles to save power on low intensity tasks, it will produce less heat as a consequence; however, depending on how long it runs on the higher or lower frequency, the thermal contribution may vary.
- `cpu_ma`: the moving average (MA) of CPU utilization. The CPU measurements might contain a lot of noise due since the method of measuring CPU utilization is usually sampling. Taking the moving average helps reduce noise in our data. The window chosen for this is 4 lags. The reasoning is that too much lag will exacerbate the effect of jumps in CPU activity, oversmoothing it.
- `temp_ma`: the moving average of device temperature. Similar to CPU usage, temperature measurements might not be very accurate. The window chosen is 4 lags as well, with the same reasoning to prevent oversmoothing.
- `temp_ma_lag`: a one interval (5s) lag from `temp_ma`. Used as a predictor since temperature is naturally autoregressive; the current temperature will be similar to the temperature sometimes close in the past.
- `temp_ma_diff`: the differences in `temp_ma`. This is used as another possible response when modelling due to the high correlation between `temp_ma` and `temp_ma_lag`.

Network activity metrics used will be the one in KBps, since their value are extremely small.

{numref}`fig-fe` and {numref}`fig-mdd` are some visualizations of the newly created features.

In [10]:
def add_lag(grp, n_lags=1):
    grp = grp.with_columns(
        pl.col("temp_ma").shift(1).alias(f"temp_ma_lag"),
    )
    return grp.drop_nulls(subset=[f"temp_ma_lag"])

df_feat = df_all.with_columns(
          pl.col("temp")
            .rolling_mean(4, center=False)
            .over(["run_id"])
            .alias("temp_ma"),
          pl.col("cpu")
            .rolling_mean(4, center=False)
            .over(["run_id"])
            .alias("cpu_ma"),
          pl.col("cpu_freq")
            .mean()
            .over(["pi_id", "run_id", "state"])
            .alias("cpu_freq_mean")
    ).with_columns(
          pl.col("temp_ma")
            .diff()
            .over(["run_id"])
            .alias("temp_ma_diff"),
)

df_model = df_feat.group_by("pi_id", "run_id").map_groups(add_lag)
df_model = df_model.drop("cpu", "cpu_freq", "time_stamp", "gpu", "net_upload_rate", "net_download_rate", "temp")

In [11]:
dev = "dev2"
plotdf = sample_section(
    df_model.filter(
        (pl.col("pi_id") == dev) & (pl.col("run_id") == chosen[dev])
    )
)
fig, ax = utils.plot_mult_time_series(
    plotdf,
    ['cpu_ma', 'cpu_freq_mean', 'temp_ma', 'temp_ma_diff'],
    suptitle=f'Resource Activity and Temperature over Time for device {dev} on run {chosen[dev]}'
)

fig.savefig("images/feat_eng.png")
plt.close(fig)

In [12]:
sns.histplot(df_feat, x='temp_ma_diff', hue='state', bins=30)

plt.gca().set_title("Distribution of temp_ma_diff")
plt.gcf().savefig("images/ma_diff_dist.png")
plt.close()

```{figure} images/feat_eng.png
:name: fig-fe
:align: center
:scale: 40

plot of samples of engineered features on `dev2`
```

```{figure} images/ma_diff_dist.png
:name: fig-mdd
:align: center
:scale: 40

distribution of `temp_ma_diff`
```

As seen from the figures above, the moving averages are smoother and less noisy, also `temp_ma_diff` being centered and symmetric, as expected since the temperature data tends to stabilize.

### Modelling

In this phase, we will attempt to build a model that can predict device temperature under specific workloads. Due to our data being highly structured, we can expect to have accurate models at the expense of generalization; our models will be good at predicting temperature under similar workloads, but might be very inaccurate for workloads that are dissimilar to these usages.

Furthermore, the model will be trained to predict the temperature moving average, this is to reduce the noise that is learned by the model that might lead to overfitting.

The feature `state` heavily impacts resource utilization, as seen it the exploration phase, but it contrains the workloads that the model can analyze and potentially reduces generalization. Hence, it will be removed from the final model; but another model purely based on `temp_ma_lag`, `pi_id`, and `state` is used as a baseline comparison model, but not used in any model that is considered.

To begin, we perform some analysis to reduce the number of features in hopes of improving model generalizations. The first feature we examine is `pi_id`. We wish to check whether there are meaningful differences between the different configurations in CPU frequency and RAM size. To do this, we perform a likelihood ratio test between a linear mixed model that uses `pi_id` and one that doesn't.

The likelihood ratio test indicates that indeed there is a difference between devices, so this feature will be kept. The details of the likelihood ratio tests are shown in {numref}`tbl-lrt`.

```{list-table} Likelihoods and LRT Statistics
:name: tbl-lrt
*   - LR Statistics
    - $\chi^2$ Degree of Freedom
    - p value
*   - 128.92
    - 3
    - <1e-16
```

In [13]:
X  = df_model.to_pandas()

In [14]:
mod0 = smf.mixedlm(
    "temp_ma ~ memory + net_upload_rate_kbps +"
    "net_download_rate_kbps + cpu_ma + cpu_freq_mean + temp_ma_lag",
    data=X, groups=X["run_id"]).fit(reml=False)
mod1 = smf.mixedlm(
    "temp_ma ~ C(pi_id) + memory + net_upload_rate_kbps +"
    "net_download_rate_kbps + cpu_ma + cpu_freq_mean + temp_ma_lag",
    data=X, groups=X["run_id"]).fit(reml=False)

In [15]:
lr_stat = 2 * (mod1.llf - mod0.llf)
df_diff = mod1.df_modelwc - mod0.df_modelwc
p_val   = chi2.sf(lr_stat, df_diff)

print("no pi_id:", mod0.llf)
print("with pi_id:", mod1.llf)
print(f"LR = {lr_stat:.2f},  df = {df_diff},  p = {p_val:e}")

no pi_id: -63367.310755461454
with pi_id: -63301.92100312747
LR = 130.78,  df = 3,  p = 3.673541e-28


We train an XGBoost regression model on a run-aware, time-ordered split of the data. Each contiguous recording (run_id) is treated as an independent block: earlier parts of the run forms the training set and the following part becomes the validation set in a walk-forward loop. Furthermore, we perform cross validation using a time series split with 6 splits and using an hour interval as validation set size.

In [16]:
tsscv = TimeSeriesSplit(n_splits=6)
xgb_mod = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    objective="reg:squarederror",
    n_jobs=-1
)
xgb_state = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    objective="reg:squarederror",
    n_jobs=-1
)

In [17]:
X = (df_model
     .drop("time", "state", "run_id", "temp_ma", "temp_ma_diff")
     .to_dummies(columns="pi_id")
    )
X_state = df_model[["temp_ma_lag", "state", "pi_id"]].to_dummies(columns=["state", "pi_id"])

y = df_model["temp_ma"]

In [18]:
mse_state = []
mse_mod = []

runs = df_model["run_id"].unique()
for run in runs:
    mse1 = []
    mse2 = []
    run_idx = np.where(df_model["run_id"] == run)[0]
    for train_idx, val_idx in tsscv.split(run_idx):
        xgb_mod.fit(X[train_idx], y[train_idx])
        xgb_state.fit(X_state[train_idx], y[train_idx])
        y_preds_mod = xgb_mod.predict(X[val_idx])
        y_preds_state = xgb_state.predict(X_state[val_idx])
        mse1.append(mse(y_preds_mod, y[val_idx]))
        mse2.append(mse(y_preds_state, y[val_idx]))
    mse_mod.append(mse1)
    mse_state.append(mse2)

gc.collect()

290840

In [20]:
# check losses

mse_mod = np.array(mse_mod)
mse_state = np.array(mse_state)

print("mod:", mse_mod.mean())
print("state:", mse_state.mean())

mod: 0.06625280026019666
state: 0.07165209367348437


The following tables shows the feature importances for the resource model and the state model when predicting `temp_ma`. The average MSE over all cross validation steps is 0.177 and 0.13 for the resource model and the state model respectively. However, both models show that `temp_ma_lag` is the dominant feature while all other features have little importance scores. This might be because stark temperature changes happen less often compared to slow temperature changes, which means the lagged moving average will be a very good predictor. For example, if the temperature have stabilized because the state hasn't changed for a while, the lagged temperature is approximately equal to the current temperature.

To attempt to account for the strong correlation between the lag and the current temperature moving average and extract more predictive power for resource activity data we train two more models with both setups again, but predicting `temp_ma_diff` instead.

In [34]:
pd.DataFrame(xgb_mod.feature_importances_,
             columns=["Feature Name"],
             index=xgb_mod.feature_names_in_)

,Feature Name
memory,0.000110
pi_id_dev0,0.000000
pi_id_dev1,0.000000
pi_id_dev2,0.000000
pi_id_dev3,0.000000
net_upload_rate_kbps,0.000072
net_download_rate_kbps,0.000052
cpu_ma,0.089091
cpu_freq_mean,0.011228
temp_ma_lag,0.899446


```{table} resource model feature importances when predicting temperature MA
:name: rm-tma
| Feature Name             | Value     |
|--------------------------|:---------:|
| `memory`                   |   0.000110 |
| `pi_id_dev0`               |   0.000000 |
| `pi_id_dev1`               |   0.000000 |
| `pi_id_dev2`               |   0.000000 |
| `pi_id_dev3`               |   0.000000 |
| `net_upload_rate_kbps`     |   0.000072 |
| `net_download_rate_kbps`   |   0.000052 |
| `cpu_ma`                   |   0.089091 |
| `cpu_freq_mean`            |   0.011228 |
| `temp_ma_lag`              |   0.899446 |

```
```{table} state model feature importances when predicting temperature MA
:name: sm-tma
| Feature Name                |     Value |
|-----------------------------|:---------:|
| `temp_ma_lag`                 |   0.915936 |
| `state_augmented_reality`     |   0.000320 |
| `state_game`                  |   0.004800 |
| `state_idle`                  |   0.008379 |
| `state_mining`                |   0.070512 |
| `state_stream`                |   0.000051 |
| `pi_id_dev0`                  |   0.000000 |
| `pi_id_dev1`                  |   0.000000 |
| `pi_id_dev2`                  |   0.000000 |
| `pi_id_dev3`                  |   0.000000 |

```

In [35]:
pd.DataFrame(xgb_state.feature_importances_,
             columns=["Feature Name"],
             index=xgb_state.feature_names_in_)

,Feature Name
temp_ma_lag,0.915936
state_augmented_reality,0.000320
state_game,0.004800
state_idle,0.008379
state_mining,0.070512
state_stream,0.000051
pi_id_dev0,0.000000
pi_id_dev1,0.000000
pi_id_dev2,0.000000
pi_id_dev3,0.000000


In [23]:
y = df_model["temp_ma_diff"]

In [24]:
xgb_mod_ = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    objective="reg:squarederror",
    n_jobs=-1
)
xgb_state_ = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    objective="reg:squarederror",
    n_jobs=-1
)

mse_state_ = []
mse_mod_ = []

runs = df_model["run_id"].unique()
for run in runs:
    mse1 = []
    mse2 = []
    run_idx = np.where(df_model["run_id"] == run)[0]
    for train_idx, val_idx in tsscv.split(run_idx):
        xgb_mod_.fit(X[train_idx], y[train_idx])
        xgb_state_.fit(X_state[train_idx], y[train_idx])
        y_preds_mod = xgb_mod.predict(X[val_idx])
        y_preds_state = xgb_state.predict(X_state[val_idx])
        mse1.append(mse(y_preds_mod, y[val_idx]))
        mse2.append(mse(y_preds_state, y[val_idx]))
    mse_mod_.append(mse1)
    mse_state_.append(mse2)

gc.collect()

113606

In [25]:
# check losses

mse_mod_ = np.array(mse_mod_)
mse_state_ = np.array(mse_state_)

print("mod:", mse_mod_.mean())
print("state:", mse_state_.mean())

mod: 2164.7372725637338
state: 2164.5345501816064


The following are the feature importances for the new models. The average MSE across cross validation runs are 0.0434 and 0.0488, respectively. The lower MSE is due to the differences in temperature MA are small. 

In [37]:
pd.DataFrame(xgb_mod_.feature_importances_,
             columns=["Feature Name"],
             index=xgb_mod.feature_names_in_)

,Feature Name
memory,0.137160
pi_id_dev0,0.000000
pi_id_dev1,0.000000
pi_id_dev2,0.000000
pi_id_dev3,0.000000
net_upload_rate_kbps,0.064952
net_download_rate_kbps,0.022611
cpu_ma,0.184090
cpu_freq_mean,0.226223
temp_ma_lag,0.364965


In [36]:
pd.DataFrame(xgb_state_.feature_importances_,
             columns=["Feature Name"],
             index=xgb_state.feature_names_in_)

,Feature Name
temp_ma_lag,0.331357
state_augmented_reality,0.203710
state_game,0.110747
state_idle,0.084816
state_mining,0.244647
state_stream,0.024724
pi_id_dev0,0.000000
pi_id_dev1,0.000000
pi_id_dev2,0.000000
pi_id_dev3,0.000000


```{table} resource model feature importances when predicting temperature MA difference
:name: rm-mad
| Feature Name             |     Value |
|--------------------------|:---------:|
| `memory`                   |   0.137160 |
| `pi_id_dev0`               |   0.000000 |
| `pi_id_dev1`               |   0.000000 |
| `pi_id_dev2`               |   0.000000 |
| `pi_id_dev3`               |   0.000000 |
| `net_upload_rate_kbps`     |   0.064952 |
| `net_download_rate_kbps`   |   0.022611 |
| `cpu_ma`                   |   0.184090 |
| `cpu_freq_mean`            |   0.226223 |
| `temp_ma_lag`              |   0.364965 |


```
```{table} state model feature importances when predicting temperature MA difference
:name: sm-mad
| Feature Name               |     Value |
|----------------------------|:---------:|
| `temp_ma_lag`                |   0.331357 |
| `state_augmented_reality`    |   0.203710 |
| `state_game`                 |   0.110747 |
| `state_idle`                |   0.084816 |
| `state_mining`               |   0.244647 |
| `state_stream`              |   0.024724 |
| `pi_id_dev0`                 |   0.000000 |
| `pi_id_dev1`                 |   0.000000 |
| `pi_id_dev2`                 |   0.000000 |
| `pi_id_dev3`                 |   0.000000 |
```

It is interesting to note, however, all XGBoost models doesn't seem to think that differences in device is important for prediction. Meanwhile the likelihood ratio test shows a different result. A hypothesis is that the differences in devices are sufficiently  modelled by other features such as CPU frequency and memory usage.

## Summary

In our analysis, we uncovered patterns in the data relating workloads, i.e. streaming, mining, etc. with resource utilization; the same workloads utilize approximately the same amount of resources. Furthermore, patterns in temperature can also be seen within each workload, temperature rises or drops quickly and stabilizes. However, temperature and CPU data measurements are noisy, motivating us to take moving averages and predict temperature moving average instead. In the modelling phase, the mixed linear model that we used points that there are significant effect of devices specifications. However, the XGBoost models doesn't seem to agree and we see the feature importance for device differences being zero. Finally, we observe low MSE achieved by the models, but this might be due to the fact that temperatures are more often 'close' together, causing either the lag to be approximately equal to the current temperature, or the difference to be approximately zero.

Future work could address this over-smoothing effect by isolating ranges of temperature rises and drops and models based on that instead.